# ⬢ Deploying Decentralized OpenClaw on Nosana Grid
### Technical Guide for Distributed AI Agent Infrastructure

This guide provides a comprehensive technical walkthrough for deploying **OpenClaw** agents onto the **Nosana Grid**. You will learn how to leverage decentralized GPU nodes to run autonomous AI workloads without traditional cloud provider constraints (latency, censorship, or high costs).

--- 

**`[Prerequisites]`**
- **Nosana Dashboard Access:** Sign up at [dashboard.nosana.com](https://dashboard.nosana.com).
- **Solana Wallet:** A compatible wallet (e.g., Phantom, Solflare) with a small amount of **SOL** for transaction fees.
- **NOS Tokens:** Used to pay for GPU compute time on the grid.
- **API Credentials:** (Optional) API tokens for external LLMs like Claude or OpenAI if using hybrid mode.

**`[Note]`**
Nosana uses a market-driven approach. You define your hardware requirements (GPU, VRAM), and the network matches you with a provider globally.

## ☰ Table of Contents
1. [System Architecture](#1.-System-Architecture)
2. [Configuring the Job Definition](#2.-Configuring-the-Job-Definition)
3. [Model Selection & VRAM Optimization](#3.-Model-Selection-&-VRAM-Optimization)
4. [The Deployment Process](#4.-The-Deployment-Process)
5. [Real-time Monitoring & Log Analysis](#5.-Real-time-Monitoring-&-Log-Analysis)
6. [Gateway Connectivity & Authentication](#6.-Gateway-Connectivity-&-Authentication)
7. [External API Integration (Hybrid Mode)](#7.-External-API-Integration)
8. [Telegram Bot Integration](#8.-Telegram-Bot-Integration)
9. [Programmatic Interaction (Python)](#9.-Programmatic-Interaction)
10. [Troubleshooting & Best Practices](#10.-Troubleshooting-&-Best-Practices)

## 1. System Architecture

The OpenClaw ecosystem on Nosana follows a modular, distributed pattern:

1. **User Client:** Interact via Telegram, Web UI, or API calls.
2. **OpenClaw Agent Container:** Orchestrates tasks, manages memory, and handles tool-calling logic. It runs as a containerized workload on the Nosana Grid.
3. **Inference Engine:** Usually **vLLM** or **Ollama** running alongside or inside the container to serve the LLM (e.g., GLM-4, Llama 3).
4. **Nosana Grid:** The decentralized network of GPU providers (Nodes) that host your deployment.

<img src="assets/architecture_diagram.png" width="700px" style="margin-top: 10px; border: 1px solid #ddd; padding: 5px;">
*Figure 1: High-level infrastructure flow between the user and the decentralized GPU provider.*

## 2. Configuring the Job Definition

In Nosana, every deployment is defined by a **Job Definition**. When you use the OpenClaw template, the following technical parameters are pre-configured, but can be modified for specialized use cases:

- **Container Image:** `openclaw/agent:latest` (or your custom fork).
- **Resources:** Defines the minimum CPU cores, System RAM, and GPU specs required.
- **Environment Variables:** Used for secrets like `API_KEYS`, `GATEWAY_TOKEN`, and `BOT_TOKEN`.
- **Exposed Ports:** Port `8080` is standard for the OpenClaw API gateway.

## 3. Model Selection & VRAM Optimization

Choosing the right model is critical for performance and cost-efficiency. **VRAM (Video RAM)** is the primary bottleneck.

**`[GPU Specification Matrix]`**
| Model | Parameter Count | Quantization | VRAM Required | Min GPU Recommendation |
| :--- | :--- | :--- | :--- | :--- |
| **GLM-4-9B-Chat** | 9B | FP16 / 4-bit | 18GB / 6GB | RTX 3090 (FP16) / RTX 3060 (4-bit) |
| **Llama-3-8B** | 8B | 4-bit (GGUF) | 8GB | RTX 3060 / 4060 |
| **DeepSeek-V3** | 671B | MoE (Mixed) | 160GB+ | 2x or 4x A100/H100 Cluster |

**`[Pro Tip]`** Use **Quantized models (4-bit or 8-bit)** to run larger models on consumer-grade GPUs without significant intelligence loss.

<img src="assets/model_selection.png" width="600px">
*Figure 2: Selecting the appropriate model and GPU node in the Nosana Dashboard.*

## 4. The Deployment Process

Follow these steps for a successful deployment:

1. **Access the Dashboard:** Go to [dashboard.nosana.com/market](https://dashboard.nosana.com/market).
2. **Template Selection:** Locate and select the **OpenClaw** template.
3. **Secrets Configuration:** Add your `GATEWAY_TOKEN` (a custom password you create) and any external API keys in the **Secrets** section. **Never paste keys in plain text fields.**
4. **Node Matching:** Select a GPU node from the marketplace. Pay attention to the **Price per Hour** and **Node Reputation**.
5. **Execution:** Click **Deploy**. The network will lock the required NOS tokens and begin the container orchestration.

<img src="assets/deployment_panel.png" width="600px">
*Figure 3: Configuring secrets and selecting a node for deployment.*

## 5. Real-time Monitoring & Log Analysis

Once the deployment is active, monitor the **Live Logs** to ensure the model loads correctly into VRAM.

**`[Key Log Indicators]`**
- `[SYSTEM] Pulling Image...` -> Container is downloading from Docker Hub.
- `[CUDA] Device found: NVIDIA GeForce RTX 4090` -> The GPU is successfully passed to the container.
- `[MODEL] Loading weights...` -> Model is being moved from disk to VRAM. This can take several minutes for large models.
- `[READY] Server listening on 0.0.0.0:8080` -> Your agent is online and ready for requests.

**Warning:** If you see `CUDA Out of Memory`, your selected model is too large for the node's VRAM. Stop the deployment and choose a smaller model or a node with more VRAM.

## 6. Gateway Connectivity & Authentication

Nosana provides a secure **Gateway URL** for every deployment. Access is restricted via the token you defined in step 4.

1. **Copy Endpoint:** Find the URL (e.g., `https://node-xxx.nosana.ci`) in the dashboard.
2. **Authenticate:** When opening the URL in a browser or connecting via API, use your `GATEWAY_TOKEN` in the Authorization header.

<img src="assets/auth_screen.png" width="500px">
*Figure 4: Authenticating with the OpenClaw Gateway.*

## 7. External API Integration (Hybrid Mode)

OpenClaw supports **Hybrid Intelligence**. You can use a local model on Nosana for basic tasks and "fallback" to high-tier models like Claude 3.5 Sonnet for complex reasoning.

**`[Configuration]`**
In the OpenClaw config file or environment variables, add:
- `CLAUDE_API_KEY`: `sk-ant-xxx`
- `OPENAI_API_KEY`: `sk-proj-xxx`

This allows the agent to switch models dynamically based on the task complexity.

## 8. Telegram Bot Integration

Run your AI agent as a Telegram bot for easy access on the go.

1. **Create Bot:** Message [@BotFather](https://t.me/botfather) and get your **HTTP API Token**.
2. **Inject Secret:** Add `TELEGRAM_BOT_TOKEN` to your Nosana deployment secrets.
3. **Set Permissions:** (Optional) Add `ALLOWED_TELEGRAM_USER_IDS` to ensure only you can use the bot.
4. **Start:** Send `/start` to your bot. It will now process messages using the GPU on the Nosana Grid.

<img src="assets/telegram_pairing.png" width="500px">
*Figure 5: Successfully pairing the OpenClaw agent with a Telegram bot.*

## 9. Programmatic Interaction (Python)

Use the script below to interact with your decentralized agent programmatically. This is ideal for integrating the agent into larger automation pipelines.

In [ ]:
import requests
import json
import sys

# --- Configuration ---
# Replace with your actual Nosana Gateway URL and Token
GATEWAY_URL = "https://your-deployment-endpoint.nosana.ci"
GATEWAY_TOKEN = "your-custom-gateway-token"
MODEL_NAME = "glm-4-flash"  # Ensure this matches the model loaded on the node

def query_agent(prompt, stream=False):
    """Sends a prompt to the OpenClaw agent on Nosana Grid."""
    endpoint = f"{GATEWAY_URL}/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {GATEWAY_TOKEN}"
    }
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "stream": stream
    }
    
    try:
        response = requests.post(endpoint, headers=headers, json=payload, stream=stream, timeout=60)
        response.raise_for_status()
        
        if stream:
            for line in response.iter_lines():
                if line:
                    # OpenClaw follows OpenAI stream format: 'data: {...}'
                    decoded_line = line.decode('utf-8')
                    if decoded_line.startswith('data: '):
                        data = json.loads(decoded_line[6:])
                        content = data['choices'][0]['delta'].get('content', '')
                        print(content, end='', flush=True)
        else:
            result = response.json()
            return result['choices'][0]['message']['content']
            
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to agent: {e}")
        return None

# Example usage:
if __name__ == "__main__":
    user_prompt = "Write a short summary of the benefits of decentralized AI compute."
    print(f"--- Querying Agent on Nosana Grid ---\nPrompt: {user_prompt}\n")
    
    # For non-streaming output:
    # response = query_agent(user_prompt)
    # print(f"Response: {response}")
    
    # For streaming output (recommended for long responses):
    query_agent(user_prompt, stream=True)

## 10. Troubleshooting & Best Practices

**`[Common Resolutions]`**
- **Model Download Failures:** Ensure the GPU Node has a stable internet connection (usually verified by the Nosana benchmark). If it fails, try a different node.
- **High Latency:** Select nodes geographically closer to you or with higher bandwidth ratings.
- **Job Suspension:** If your NOS balance runs out, the job will stop immediately. Enable **Auto-Top-up** in the dashboard settings.

**`[Security Best Practices]`**
- **Secret Management:** Use the Nosana Secrets dashboard for all keys. Never hardcode them in your local scripts if you plan to share them.
- **Rate Limiting:** If deploying a public bot, implement rate limiting in the OpenClaw configuration to manage NOS costs.
- **Node Selection:** Prioritize nodes with a high **Stability Score** to avoid unexpected downtime.

--- 

**`[Further Reading]`**
- [Nosana SDK Documentation](https://learn.nosana.com/)
- [OpenClaw Framework Repository](https://github.com/openclaw/openclaw)
- [Nosana Grid Status](https://status.nosana.com/)
- [Solana Explorer](https://explorer.solana.com/) (Track your NOS/SOL transactions)